# Optimise model (nested CV)

Run grid search with cross-validation for hyperparameter optimisation.
For each outer fold, run inner CV grid search on the outer training set.

In [1]:
run = "k"

In [2]:
import sys
sys.path.append("../../../src/ml")

In [3]:
!pwd

/home/mu2ecrv/sgrant-ana/mu2e-cosmic


In [ ]:
from assemble import AssembleDataset
assembler = AssembleDataset(run=run, cutset_name="MLPreprocess", verbosity=2)
data = assembler.assemble_dataset(n_outer_folds=5)

In [ ]:
from optimise import Optimise

param_grid = {
    "n_estimators": [100, 200, 500, 1000],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.05, 0.1, 0.3],
}
fixed_params = {"device": "cuda", "tree_method": "hist"}

# Nested CV: run inner CV grid search on each outer fold's training set
outer_best_hps = []
for k, (train_idx, test_idx) in enumerate(data["outer_folds"]):
    print(f"\n{'='*60}")
    print(f"Outer fold {k}")
    print(f"{'='*60}")
    fold_data = AssembleDataset.get_fold_data(data, train_idx, test_idx)
    opt = Optimise(fold_data, run=run, min_efficiency=0.999, scale_features=False)
    best = opt.grid_search_cv(param_grid, n_folds=5, fixed_params=fixed_params)
    outer_best_hps.append(best["hyperparams"])
    print(f"Fold {k} best: {best['hyperparams']}, deadtime: {best['mean_deadtime']*100:.3f}%")

In [ ]:
import pandas as pd
from collections import Counter

# Summarise best HPs across outer folds
print("Best hyperparameters per outer fold:")
for k, hp in enumerate(outer_best_hps):
    print(f"  Fold {k}: {hp}")

# Check consistency
hp_tuples = [tuple(sorted(hp.items())) for hp in outer_best_hps]
hp_counts = Counter(hp_tuples)
print(f"\nMost common: {dict(hp_counts.most_common(1)[0][0])} ({hp_counts.most_common(1)[0][1]}/{len(outer_best_hps)} folds)")

# Use the most common HP set as the consensus
consensus_hp = dict(hp_counts.most_common(1)[0][0])
print(f"\nConsensus best HP: {consensus_hp}")

In [ ]:
# The last opt instance still has its summary from the final outer fold
# For a full picture, run the diagnostic plots on the last fold as a representative example
opt.plot_overfit_diagnostic("max_depth")
opt.plot_overfit_diagnostic("n_estimators")
opt.plot_overfit_diagnostic("learning_rate")
opt.save_summary(tag="nested_cv")

## Analysis

Check whether the consensus best HP is consistent across outer folds. If all folds agree, the optimisation is robust. If they disagree, the landscape is flat and the choice doesn't matter much. Best way is to pick the simplest model.

## Class imbalance 

In [ ]:
data["y"].value_counts()

In [ ]:
param_grid_spw = {
    "scale_pos_weight": [1, 10, 50, 90],
}
fixed_params_spw = {**consensus_hp, "device": "cuda", "tree_method": "hist"}

# Use a single outer fold for the SPW scan (it doesn't affect ranking)
train_idx, test_idx = data["outer_folds"][0]
fold_data = AssembleDataset.get_fold_data(data, train_idx, test_idx)
opt_spw = Optimise(fold_data, run=run, min_efficiency=0.999, scale_features=False)
best_spw = opt_spw.grid_search_cv(param_grid_spw, n_folds=5, fixed_params=fixed_params_spw)

In [ ]:
opt_spw.plot_overfit_diagnostic("scale_pos_weight")

In [ ]:
opt_spw.get_summary()

In [ ]:
from train import Train
from validate import Validate

scale_pos_weight = 90 

hp = {**consensus_hp, "scale_pos_weight": scale_pos_weight}

# Train on first outer fold to check score distribution
train_idx, test_idx = data["outer_folds"][0]
fold_data = AssembleDataset.get_fold_data(data, train_idx, test_idx)

trn = Train(fold_data, run=run, verbosity=0)
results = trn.train(tag="spw_check", save_output=False, **hp)

val = Validate(results, run=run, verbosity=0)
val.roc_auc()
val.find_threshold(min_eff=0.999, plot=True, show=True)

In [ ]:
thr = val.find_threshold(min_eff=0.999, plot=False, show=False)
val.plot_score_distribution(threshold=thr["threshold"])